In [8]:
!sudo apt-get install -y zstd
# Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Start Ollama in background using subprocess
import subprocess, time
proc = subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(5)  # Wait for startup

# Pull the model
!ollama pull gemma3:12b
# !ollama pull gpt-oss:20b

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
zstd is already the newest version (1.4.8+dfsg-3build1).
0 upgraded, 0 newly installed, 0 to remove and 133 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%               67.9%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.
]11;?\pulling manifest ⠋ pulling manifest ⠙ pulling manifest ⠹ pulling manifest ⠸ pulling manifest ⠼ pulling manifest ⠴ pulling manifest ⠦ pulling manifest 
pulling e8ad13eff07a:   0% ▕                  ▏ 3.9 MB/8.1 GB                  pulling manifest 
pulling e8ad13eff07a:   1% ▕                  ▏  72 MB/8

In [3]:
!ollama list

]11;?\NAME           ID              SIZE      MODIFIED           
gpt-oss:20b    17052f91a42e    13 GB     About a minute ago    
qwen3-vl:8b    901cae732162    6.1 GB    3 minutes ago         


In [11]:
import json
import os
import re
import time
import logging
import queue
import threading
import traceback
from datetime import datetime
from tqdm import tqdm
from collections import deque
from PIL import Image
import requests
import base64
from io import BytesIO

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('ollama_meme_english.log'),
        logging.StreamHandler()
    ]
)

# -------------------------
# Ollama API Manager
# -------------------------
class OllamaApiManager:
    """
    Manages Ollama API calls with rate limiting and retry logic.
    """

    def __init__(self, base_url="http://localhost:11434", model="gemma3:12b", rate_limit_delay=2):
        self.base_url = base_url.rstrip('/')
        self.model = model
        self.rate_limit_delay = rate_limit_delay
        self.last_call_time = 0
        self.call_count = 0
        
        logging.info(f"Ollama API Manager initialized: {base_url}, model={model}")
        
        # Verify Ollama is running and model is available
        self._verify_setup()
    
    def _verify_setup(self):
        """Verify Ollama is running and the model is available."""
        try:
            response = requests.get(f"{self.base_url}/api/tags", timeout=10)
            if response.status_code == 200:
                models = response.json().get("models", [])
                model_names = [m.get("name", "") for m in models]
                logging.info(f"Available models: {model_names}")
                
                if self.model not in model_names:
                    logging.warning(f"Model '{self.model}' not found. Attempting to pull...")
                    self._pull_model()
            else:
                logging.error(f"Failed to connect to Ollama. Status: {response.status_code}")
                raise ConnectionError("Cannot connect to Ollama server")
        except requests.exceptions.ConnectionError:
            logging.error("Cannot connect to Ollama. Make sure Ollama is running on localhost:11434")
            raise ConnectionError("Ollama is not running or not accessible")
    
    def _pull_model(self):
        """Pull the model if not available."""
        try:
            logging.info(f"Pulling model '{self.model}'... This may take a while.")
            response = requests.post(
                f"{self.base_url}/api/pull",
                json={"name": self.model},
                stream=True,
                timeout=3600
            )
            if response.status_code == 200:
                for line in response.iter_lines():
                    if line:
                        status = json.loads(line.decode('utf-8'))
                        if 'status' in status:
                            logging.info(f"Pull status: {status['status']}")
                logging.info(f"Model '{self.model}' pulled successfully")
            else:
                logging.error(f"Failed to pull model. Status: {response.status_code}")
        except Exception as e:
            logging.error(f"Error pulling model: {e}")
            raise

    def generate_content(self, prompt, image=None, timeout=120, options=None):
        """
        Make an API call to Ollama with optional image support using /api/generate endpoint.
        """
        # Rate limiting
        elapsed = time.time() - self.last_call_time
        if elapsed < self.rate_limit_delay:
            time.sleep(self.rate_limit_delay - elapsed)

        payload = {
            "model": self.model,
            "prompt": prompt,
            "stream": False,
            "keep_alive": -1,  # Keep model loaded
            "options": options or {
                "num_ctx": 8192,
                "temperature": 0.1,
                "num_gpu": 99,
                "num_predict": 1024,  # Cap output to prevent verbose rambling
                "min_p": 0.05,
                "repeat_penalty": 1.1
            }
        }

        # Add image if provided
        if image is not None:
            try:
                if isinstance(image, str):
                    image = Image.open(image)

                # Resize image for processing (max 768x768 for T4 GPU)
                image.thumbnail((768, 768), Image.Resampling.LANCZOS)

                # Convert palette/RGBA images to RGB before saving as JPEG
                if image.mode in ('P', 'RGBA', 'LA', 'PA'):
                    image = image.convert('RGB')

                # Convert to base64
                buffered = BytesIO()
                image.save(buffered, format="JPEG", quality=85)
                img_base64 = base64.b64encode(buffered.getvalue()).decode('utf-8')
                payload["images"] = [img_base64]
            except Exception as e:
                logging.error(f"Failed to process image: {e}")
                raise

        # Make the API call
        try:
            response = requests.post(
                f"{self.base_url}/api/generate",
                json=payload,
                timeout=timeout
            )

            if response.status_code == 200:
                result = response.json()
                self.last_call_time = time.time()
                self.call_count += 1
                response_text = result.get("response", "")

                # For thinking models, use thinking field if response is empty
                if not response_text and result.get("thinking"):
                    response_text = result["thinking"]

                logging.info(f"Ollama returned {len(response_text)} chars, done={result.get('done', False)}")
                if len(response_text) == 0:
                    logging.warning(f"Full Ollama response: {result}")
                return response_text
            else:
                raise Exception(f"Ollama API error: {response.status_code} - {response.text}")

        except requests.exceptions.Timeout:
            raise TimeoutError(f"Ollama request timed out after {timeout}s")
        except Exception as e:
            logging.error(f"Ollama API call failed: {e}")
            raise
    
    def get_usage_stats(self):
        """Get usage statistics."""
        return {
            "total_calls": self.call_count,
            "model": self.model,
            "base_url": self.base_url
        }


# -------------------------
# Meme Evaluator (English)
# -------------------------
class MemeEvaluator:
    """
    Zero-shot political meme evaluator using Ollama (gemma3:12b) multimodal API.
    Evaluates each question individually for faster processing.
    """

    def __init__(self, api_manager: OllamaApiManager, model_name="gemma3:12b",
                 image_dir=".", language="en"):
        self.api_manager = api_manager
        self.model_name = model_name
        self.image_dir = image_dir
        self.language = language  # "en" for English
        logging.info(f"MemeEvaluator initialized: model={model_name}, lang={language}, image_dir={image_dir}")

    def _find_image_path(self, image_id):
        """
        Resolve the image path for a given image_id.
        Checks for .jpg, .jpeg, and .png extensions (case-insensitive).
        """
        extensions = ['.jpg', '.jpeg', '.png', '.JPG', '.JPEG', '.PNG']
        for ext in extensions:
            path = os.path.join(self.image_dir, f"{image_id}{ext}")
            if os.path.exists(path):
                return path

        # Also try lowercase image_id
        for ext in extensions:
            path = os.path.join(self.image_dir, f"{image_id.lower()}{ext}")
            if os.path.exists(path):
                return path

        return None

    def _build_single_question_prompt(self, question):
        """Build a prompt for a single question."""
        q_id = question["id"]
        q_type = question["type"]
        q_text = question["question"][self.language]

        if q_type == "mcq":
            options = question["options"][self.language]
            options_str = ", ".join([f"{k}) {v}" for k, v in sorted(options.items())])
            prompt = (
                f"You are a political meme expert. Analyze the image carefully.\n\n"
                f"Question {q_id}: {q_text}\n"
                f"Options: {options_str}\n\n"
                f"Answer with a SINGLE LETTER only (A, B, C, D, E, or F). "
                f"No other text. No explanation."
            )
        else:
            prompt = (
                f"You are a political meme expert. Analyze the image carefully.\n\n"
                f"Question {q_id}: {q_text}\n\n"
                f"Answer in 1-3 sentences maximum. "
                f"Be direct and concise. "
                f"State your answer immediately. Do NOT explain your reasoning process. "
                f"Do NOT use phrases like 'let me think', 'let's analyze', 'wait', 'I need to check'. "
                f"Base your answer on what you see in the image."
            )

        return prompt

    def _clean_thinking_prefix(self, text):
        """Remove thinking/reasoning prefixes from model responses."""
        if not text:
            return text

        text = text.strip()

        # Pattern 1: Remove "Got it, let's analyze..." type intros
        text = re.sub(r'^(?:got it,?\s*)?let\'s\s+(?:analyze|look at|break down|read|translate)\s+the\s+', '', text, flags=re.IGNORECASE)

        lines = text.strip().split("\n")

        # Detect if this is a verbose internal monologue
        thinking_patterns = [
            r'\bwait,\s*(?:maybe|no|the)',
            r'\blet\'s\s+(?:think|check|analyze|translate)',
            r'\bi\'m\s+(?:not sure|overcomplicating|stuck)',
            r'\bi think the answer is',
            r'\btranslat(?:e|ing)',
            r'\bbreaking it down',
            r'\bkey is',
        ]

        thinking_score = sum(1 for line in lines for pat in thinking_patterns if re.search(pat, line, re.IGNORECASE))

        # If there's significant thinking, find the actual answer
        if thinking_score >= 3:
            # Look for the final answer/conclusion part
            # Usually starts with phrases like "Yes, the image...", "No, the image...", "The image..."
            answer_start_patterns = [
                r'^(yes|no),\s+the\s+image\s+',
                r'^this\s+image\s+is\s+',
                r'^the\s+image\s+(?:references|shows|does\s+not|is\s+a|relates)',
                r'^the\s+correct\s+answer\s+is\s+',
                r'^the\s+answer\s+is\s*:\s*',
                r'^in\s+(?:bangladesh|the\s+context)',
                r'^based\s+on\s+the\s+image',
            ]

            # Find the LAST occurrence of an answer-like line
            # (since the model often reiterates and eventually concludes)
            for i in range(len(lines) - 1, -1, -1):
                stripped = lines[i].strip()
                if any(re.search(pat, stripped, re.IGNORECASE) for pat in answer_start_patterns):
                    # Return from this line onward
                    return "\n".join(lines[i:]).strip()

            # If no clear answer pattern, try to find a line that looks like a direct statement
            for i in range(len(lines) - 1, -1, -1):
                stripped = lines[i].strip()
                if len(stripped) > 20 and not re.match(r'^(wait|let\'s|i\'m|i think|translat|got it|not sure)', stripped, re.IGNORECASE):
                    return stripped

            # Last resort: return the last non-empty line
            for i in range(len(lines) - 1, -1, -1):
                if lines[i].strip():
                    return lines[i].strip()

        return text.strip()

    def _parse_response(self, raw_text):
        """
        Parse the model's response to extract per-question answers.
        Expects a JSON object with keys Q1-Q7.
        """
        if not raw_text:
            return {}

        # Try to extract JSON from the response
        # The model might wrap it in ```json ... ``` or similar
        json_match = re.search(r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}', raw_text, re.DOTALL)
        if json_match:
            try:
                parsed = json.loads(json_match.group())
                return parsed
            except json.JSONDecodeError:
                pass

        # Try stripping markdown code fences
        cleaned = re.sub(r'```(?:json)?\s*', '', raw_text)
        cleaned = re.sub(r'```\s*$', '', cleaned).strip()
        try:
            parsed = json.loads(cleaned)
            return parsed
        except json.JSONDecodeError:
            pass

        # Fallback: try to extract individual Q answers using regex
        answers = {}
        for i in range(1, 8):
            pattern = rf'"Q{i}"\s*:\s*"([^"]*)"'
            match = re.search(pattern, raw_text)
            if match:
                answers[f"Q{i}"] = match.group(1)
            else:
                # Try with single quotes
                pattern = rf"'Q{i}'\s*:\s*'([^']*)'"
                match = re.search(pattern, raw_text)
                if match:
                    answers[f"Q{i}"] = match.group(1)

        if answers:
            return answers

        logging.warning(f"Could not parse JSON response. Raw text:\n{raw_text[:500]}")
        return {"raw_response": raw_text}

    def evaluate_single_image(self, image_entry, max_retries=3):
        """
        Evaluate a single image with its questions, one question at a time.
        Returns a result dict with image_id, model answers, and correct answers.
        """
        image_id = image_entry["image_id"]
        questions = image_entry["questions"]

        # Find the image file
        image_path = self._find_image_path(image_id)
        if not image_path:
            logging.error(f"Image not found for {image_id} in {self.image_dir}")
            return {
                "image_id": image_id,
                "status": "error",
                "error": f"Image file not found for {image_id}",
                "questions": []
            }

        # Load the image
        try:
            image = Image.open(image_path)
            logging.info(f"Loaded image: {image_path} ({image.size[0]}x{image.size[1]}, {image.format})")
        except Exception as e:
            logging.error(f"Failed to load image {image_path}: {e}")
            return {
                "image_id": image_id,
                "status": "error",
                "error": f"Failed to load image: {str(e)}",
                "questions": []
            }

        # Evaluate each question individually
        question_results = []
        for q in questions:
            q_id = q["id"]
            q_type = q["type"]
            prompt = self._build_single_question_prompt(q)

            # Call the Ollama API with image + text
            raw_response = ""
            # Timeout: MCQ questions need less time, open-ended need more
            # Q7 (punchline explanation) is the most complex — give it maximum time
            if q_id == "Q7":
                q_timeout = 600  # 10 minutes for Q7
            elif q_type == "mcq":
                q_timeout = 120
            else:
                q_timeout = 300  # 5 minutes for other open-ended
            for attempt in range(1, max_retries + 1):
                try:
                    response = self.api_manager.generate_content(
                        prompt=prompt,
                        image=image_path,
                        timeout=q_timeout
                    )

                    raw_response = (response or "").strip()

                    # Only warn on truly empty responses (1-char answers like "A", "B" are valid MCQ outputs)
                    if len(raw_response) == 0:
                        logging.warning(
                            f"Empty response for {image_id}/{q_id} (attempt {attempt})"
                        )
                        if attempt < max_retries:
                            raw_response = ""
                            continue

                    break

                except Exception as e:
                    logging.error(f"API call failed for {image_id}/{q_id} (attempt {attempt}/{max_retries}): {e}")
                    if attempt == max_retries:
                        logging.warning(f"Skipping {q_id} after {max_retries} failures")
                        raw_response = ""
                    time.sleep(2 * attempt)

            # Clean up MCQ response (extract just the letter)
            if q_type == "mcq":
                cleaned = raw_response.strip().upper()
                # Extract first letter that matches an option
                for char in cleaned:
                    if char in "ABCDEF":
                        raw_response = char
                        break
            else:
                # Strip thinking prefix from open-ended responses
                raw_response = self._clean_thinking_prefix(raw_response)

            correct_answer = q["correct_answer"][self.language]

            result = {
                "question_id": q_id,
                "question_type": q_type,
                "question_text": q["question"][self.language],
                "model_answer": raw_response,
                "correct_answer": correct_answer,
            }

            # For MCQs, add options for reference
            if q_type == "mcq" and "options" in q:
                result["options"] = q["options"][self.language]

            question_results.append(result)

        return {
            "image_id": image_id,
            "status": "success",
            "questions": question_results
        }

    def evaluate_dataset(self, data, output_path, start_idx=0, end_idx=None):
        """
        Evaluate the full dataset (or a slice of it).
        Saves results incrementally and supports resuming from existing output.
        """
        if end_idx is None:
            end_idx = len(data)

        # Load existing results for resumability
        existing_results = []
        processed_ids = set()
        if os.path.exists(output_path):
            try:
                with open(output_path, 'r', encoding='utf-8') as f:
                    existing_results = json.load(f)
                processed_ids = {r["image_id"] for r in existing_results}
                logging.info(f"Resuming: found {len(processed_ids)} already-processed images in {output_path}")
            except Exception as e:
                logging.warning(f"Could not load existing results from {output_path}: {e}")

        results = list(existing_results)
        subset = data[start_idx:end_idx]

        print(f"\n{'='*60}")
        print(f"Political Meme Evaluation — English Questions")
        print(f"{'='*60}")
        print(f"Total images in dataset: {len(data)}")
        print(f"Processing range: {start_idx} to {end_idx - 1} ({len(subset)} images)")
        print(f"Already processed: {len(processed_ids)}")
        print(f"Output file: {output_path}")
        print(f"{'='*60}\n")

        new_count = 0
        error_count = 0

        for i, entry in enumerate(tqdm(subset, desc="Evaluating images")):
            image_id = entry["image_id"]

            if image_id in processed_ids:
                logging.info(f"Skipping {image_id} (already processed)")
                continue

            result = self.evaluate_single_image(entry)
            results.append(result)
            processed_ids.add(image_id)
            new_count += 1

            if result["status"] == "error":
                error_count += 1
                print(f"\n  ✗ {image_id}: {result.get('error', 'Unknown error')}")
            else:
                # Print a brief summary
                mcq_answers = []
                for qr in result["questions"]:
                    if qr["question_type"] == "mcq":
                        match = "✓" if qr["model_answer"].strip().upper() == qr["correct_answer"].strip().upper() else "✗"
                        mcq_answers.append(f"{qr['question_id']}:{match}")
                print(f"\n  ✓ {image_id}: MCQ results: {', '.join(mcq_answers)}")

            # Save intermediate results every 5 images
            if new_count % 5 == 0:
                self._save_results(results, output_path)
                logging.info(f"Saved intermediate results ({len(results)} total)")

        # Final save
        self._save_results(results, output_path)

        # Print summary
        print(f"\n{'='*60}")
        print(f"EVALUATION COMPLETE")
        print(f"{'='*60}")
        print(f"Total processed: {len(results)}")
        print(f"Newly processed: {new_count}")
        print(f"Errors: {error_count}")
        print(f"Results saved to: {output_path}")
        self._print_mcq_accuracy(results)
        print(f"{'='*60}\n")

        return results

    def _save_results(self, results, output_path):
        """Save results to a JSON file."""
        os.makedirs(os.path.dirname(output_path) if os.path.dirname(output_path) else ".", exist_ok=True)
        with open(output_path, 'w', encoding='utf-8') as f:
            json.dump(results, f, ensure_ascii=False, indent=2)

    def _print_mcq_accuracy(self, results):
        """Print MCQ accuracy summary."""
        total_mcq = 0
        correct_mcq = 0
        per_question = {}

        for r in results:
            if r["status"] != "success":
                continue
            for qr in r["questions"]:
                if qr["question_type"] == "mcq":
                    q_id = qr["question_id"]
                    if q_id not in per_question:
                        per_question[q_id] = {"total": 0, "correct": 0}
                    per_question[q_id]["total"] += 1
                    total_mcq += 1

                    if qr["model_answer"].strip().upper() == qr["correct_answer"].strip().upper():
                        correct_mcq += 1
                        per_question[q_id]["correct"] += 1

        if total_mcq > 0:
            print(f"\nMCQ Accuracy: {correct_mcq}/{total_mcq} ({correct_mcq/total_mcq*100:.1f}%)")
            for q_id in sorted(per_question.keys()):
                stats = per_question[q_id]
                acc = stats['correct'] / stats['total'] * 100 if stats['total'] > 0 else 0
                print(f"  {q_id}: {stats['correct']}/{stats['total']} ({acc:.1f}%)")


# -------------------------
# Main
# -------------------------
def main():
    """Main function to run English meme evaluation with Ollama (qwen3.5:8b) on Kaggle."""

    # === CONFIGURATION ===
    # For Kaggle: Update these paths to match your dataset input directory
    # On Kaggle, input files are typically in /kaggle/input/
    dataset_path = "/kaggle/input/datasets/godfatherthr/final-corpus/Final_Corpus/BD_Political_QA.json"

    # Directory containing the meme images (JPG/PNG)
    image_dir = "/kaggle/input/datasets/godfatherthr/final-corpus/Final_Corpus/image"

    # Output path for results (Kaggle working directory)
    output_base_path = "/kaggle/working/"
    os.makedirs(output_base_path, exist_ok=True)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    output_path = os.path.join(output_base_path, f"ollama_qwen35_results_{timestamp}.json")

    # Model to use (Ollama)
    model_name = "gemma3:12b"
    
    # Ollama server URL (change if running on a different host/port)
    # For Kaggle, you'll need to run Ollama locally in the notebook
    ollama_base_url = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")

    # === INITIALIZE OLLAMA ===
    try:
        logging.info(f"Initializing Ollama with model: {model_name}")
        api_manager = OllamaApiManager(
            base_url=ollama_base_url,
            model=model_name,
            rate_limit_delay=2  # Adjust based on your hardware
        )
        logging.info("Ollama initialized successfully")

    except Exception as e:
        logging.error(f"Failed to initialize Ollama: {e}")
        print(f"Error initializing Ollama: {e}")
        print("\nFor Kaggle setup:")
        print("1. Install Ollama: !curl -fsSL https://ollama.com/install.sh | sh")
        print("2. Start Ollama: !ollama serve &")
        print("3. Pull model: !ollama pull qwen3.5:8b")
        print("4. Then run this script")
        return

    # === LOAD DATASET ===
    try:
        with open(dataset_path, 'r', encoding='utf-8') as f:
            dataset = json.load(f)
        print(f"Loaded {len(dataset)} image entries from {dataset_path}")
    except Exception as e:
        logging.error(f"Failed to load dataset: {e}")
        print(f"Error loading dataset: {e}")
        print(f"Expected dataset at: {dataset_path}")
        print("Update dataset_path in the script to match your Kaggle input directory")
        return

    # === INITIALIZE EVALUATOR ===
    evaluator = MemeEvaluator(
        api_manager=api_manager,
        model_name=model_name,
        image_dir=image_dir,
        language="en"
    )

    # === RUN EVALUATION ===
    try:
        results = evaluator.evaluate_dataset(
            data=dataset,
            output_path=output_path,
            start_idx=0,
            end_idx=20  # Test with first 5 images only
        )

        # Print Ollama usage statistics
        usage_stats = api_manager.get_usage_stats()
        print("\nOllama Usage Statistics:")
        print(f"Total API calls made: {usage_stats['total_calls']}")
        print(f"Model: {usage_stats['model']}")
        print(f"Base URL: {usage_stats['base_url']}")
        logging.info(f"Ollama usage stats: {usage_stats}")

    except Exception as e:
        logging.error(f"Evaluation failed: {e}", exc_info=True)
        print(f"Error during evaluation: {e}")


if __name__ == "__main__":
    main()

2026-04-14 11:09:29,958 - INFO - Initializing Ollama with model: gemma3:12b
2026-04-14 11:09:29,959 - INFO - Ollama API Manager initialized: http://localhost:11434, model=gemma3:12b
2026-04-14 11:09:29,964 - INFO - Available models: ['gemma3:12b', 'gpt-oss:20b', 'qwen3-vl:8b']
2026-04-14 11:09:29,964 - INFO - Ollama initialized successfully
2026-04-14 11:09:30,034 - INFO - MemeEvaluator initialized: model=gemma3:12b, lang=en, image_dir=/kaggle/input/datasets/godfatherthr/final-corpus/Final_Corpus/image


Loaded 122 image entries from /kaggle/input/datasets/godfatherthr/final-corpus/Final_Corpus/BD_Political_QA.json

Political Meme Evaluation — English Questions
Total images in dataset: 122
Processing range: 0 to 19 (20 images)
Already processed: 0
Output file: /kaggle/working/ollama_qwen35_results_20260414_110929.json



Evaluating images:   0%|          | 0/20 [00:00<?, ?it/s]2026-04-14 11:09:30,054 - INFO - Loaded image: /kaggle/input/datasets/godfatherthr/final-corpus/Final_Corpus/image/IMAGE_001.jpg (1080x1080, JPEG)
2026-04-14 11:09:41,893 - INFO - Ollama returned 173 chars, done=True
2026-04-14 11:09:47,351 - INFO - Ollama returned 259 chars, done=True
2026-04-14 11:09:50,339 - INFO - Ollama returned 1 chars, done=True
2026-04-14 11:09:53,257 - INFO - Ollama returned 1 chars, done=True
2026-04-14 11:09:56,192 - INFO - Ollama returned 1 chars, done=True
2026-04-14 11:10:01,613 - INFO - Ollama returned 264 chars, done=True
2026-04-14 11:10:08,520 - INFO - Ollama returned 376 chars, done=True
Evaluating images:   5%|▌         | 1/20 [00:38<12:11, 38.49s/it]2026-04-14 11:10:08,552 - INFO - Loaded image: /kaggle/input/datasets/godfatherthr/final-corpus/Final_Corpus/image/IMAGE_002.jpg (2048x1534, JPEG)



  ✓ IMAGE_001: MCQ results: Q3:✓, Q4:✓, Q5:✓


2026-04-14 11:10:15,294 - INFO - Ollama returned 231 chars, done=True
2026-04-14 11:10:23,084 - INFO - Ollama returned 396 chars, done=True
2026-04-14 11:10:26,037 - INFO - Ollama returned 1 chars, done=True
2026-04-14 11:10:29,002 - INFO - Ollama returned 1 chars, done=True
2026-04-14 11:10:31,929 - INFO - Ollama returned 1 chars, done=True
2026-04-14 11:10:37,924 - INFO - Ollama returned 258 chars, done=True
2026-04-14 11:10:44,680 - INFO - Ollama returned 394 chars, done=True
Evaluating images:  10%|█         | 2/20 [01:14<11:08, 37.12s/it]2026-04-14 11:10:44,717 - INFO - Loaded image: /kaggle/input/datasets/godfatherthr/final-corpus/Final_Corpus/image/IMAGE_003.jpg (1920x1920, JPEG)



  ✓ IMAGE_002: MCQ results: Q3:✗, Q4:✗, Q5:✓


... (32 outputs left out)

2026-04-14 11:19:33,075 - INFO - Ollama returned 180 chars, done=True
2026-04-14 11:19:39,499 - INFO - Ollama returned 315 chars, done=True
2026-04-14 11:19:42,417 - INFO - Ollama returned 1 chars, done=True
2026-04-14 11:19:45,263 - INFO - Ollama returned 1 chars, done=True
2026-04-14 11:19:48,117 - INFO - Ollama returned 1 chars, done=True
2026-04-14 11:19:53,923 - INFO - Ollama returned 275 chars, done=True
2026-04-14 11:19:59,767 - INFO - Ollama returned 330 chars, done=True
Evaluating images:  95%|█████████▌| 19/20 [10:29<00:32, 32.43s/it]2026-04-14 11:19:59,785 - INFO - Loaded image: /kaggle/input/datasets/godfatherthr/final-corpus/Final_Corpus/image/IMAGE_020.jpg (2048x1857, JPEG)



  ✓ IMAGE_019: MCQ results: Q3:✓, Q4:✗, Q5:✓


2026-04-14 11:20:05,941 - INFO - Ollama returned 191 chars, done=True
2026-04-14 11:20:12,254 - INFO - Ollama returned 260 chars, done=True
2026-04-14 11:20:15,229 - INFO - Ollama returned 1 chars, done=True
2026-04-14 11:20:18,209 - INFO - Ollama returned 1 chars, done=True
2026-04-14 11:20:21,205 - INFO - Ollama returned 1 chars, done=True
2026-04-14 11:20:26,840 - INFO - Ollama returned 291 chars, done=True
2026-04-14 11:20:33,227 - INFO - Ollama returned 345 chars, done=True
2026-04-14 11:20:33,232 - INFO - Saved intermediate results (20 total)
Evaluating images: 100%|██████████| 20/20 [11:03<00:00, 33.16s/it]
2026-04-14 11:20:33,238 - INFO - Ollama usage stats: {'total_calls': 140, 'model': 'gemma3:12b', 'base_url': 'http://localhost:11434'}



  ✓ IMAGE_020: MCQ results: Q3:✗, Q4:✗, Q5:✓

EVALUATION COMPLETE
Total processed: 20
Newly processed: 20
Errors: 0
Results saved to: /kaggle/working/ollama_qwen35_results_20260414_110929.json

MCQ Accuracy: 20/60 (33.3%)
  Q3: 7/20 (35.0%)
  Q4: 2/20 (10.0%)
  Q5: 11/20 (55.0%)


Ollama Usage Statistics:
Total API calls made: 140
Model: gemma3:12b
Base URL: http://localhost:11434


In [4]:
# !cat ollama_meme_english.log | tail -30

In [12]:
import json
import os
import re
import time
import logging
import requests
from tqdm import tqdm

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler('political_evaluation.log'),
        logging.StreamHandler()
    ]
)


class OllamaApiManager:
    """
    Manages Ollama API calls with rate limiting and retry logic.
    """

    def __init__(self, base_url="http://localhost:11434", model="gpt-oss:20b", rate_limit_delay=2):
        self.base_url = base_url.rstrip('/')
        self.model = model
        self.rate_limit_delay = rate_limit_delay
        self.last_call_time = 0
        self.call_count = 0
        
        logging.info(f"Ollama API Manager initialized: {base_url}, model={model}")
        self._verify_setup()
    
    def _verify_setup(self):
        """Verify Ollama is running and the model is available."""
        try:
            response = requests.get(f"{self.base_url}/api/tags", timeout=10)
            if response.status_code == 200:
                models = response.json().get("models", [])
                model_names = [m.get("name", "") for m in models]
                logging.info(f"Available models: {model_names}")
                
                if self.model not in model_names:
                    logging.warning(f"Model '{self.model}' not found. Attempting to pull...")
                    self._pull_model()
            else:
                raise ConnectionError("Cannot connect to Ollama server")
        except requests.exceptions.ConnectionError:
            logging.error("Cannot connect to Ollama. Make sure Ollama is running on localhost:11434")
            raise ConnectionError("Ollama is not running or not accessible")
    
    def _pull_model(self):
        """Pull the model if not available."""
        try:
            logging.info(f"Pulling model '{self.model}'... This may take a while.")
            response = requests.post(
                f"{self.base_url}/api/pull",
                json={"name": self.model},
                stream=True,
                timeout=3600
            )
            if response.status_code == 200:
                for line in response.iter_lines():
                    if line:
                        status = json.loads(line.decode('utf-8'))
                        if 'status' in status:
                            logging.info(f"Pull status: {status['status']}")
                logging.info(f"Model '{self.model}' pulled successfully")
            else:
                logging.error(f"Failed to pull model. Status: {response.status_code}")
        except Exception as e:
            logging.error(f"Error pulling model: {e}")
            raise

    def generate_content(self, prompt, timeout=300, options=None):
        """Make an API call to Ollama."""
        elapsed = time.time() - self.last_call_time
        if elapsed < self.rate_limit_delay:
            time.sleep(self.rate_limit_delay - elapsed)
        
        payload = {
            "model": self.model,
            "prompt": prompt,
            "stream": False,
            "options": options or {
                "num_ctx": 32768,
                "temperature": 0.1,
                "num_gpu": 99,
                "think": False
            }
        }
        
        try:
            response = requests.post(
                f"{self.base_url}/api/generate",
                json=payload,
                timeout=timeout
            )
            
            if response.status_code == 200:
                result = response.json()
                self.last_call_time = time.time()
                self.call_count += 1
                response_text = result.get("response", "")

                # Fallback for thinking models
                if not response_text and result.get("thinking"):
                    response_text = result["thinking"]

                return response_text
            else:
                raise Exception(f"Ollama API error: {response.status_code} - {response.text}")
                
        except requests.exceptions.Timeout:
            raise TimeoutError(f"Ollama request timed out after {timeout}s")
        except Exception as e:
            logging.error(f"Ollama API call failed: {e}")
            raise
    
    def get_usage_stats(self):
        """Get usage statistics."""
        return {
            "total_calls": self.call_count,
            "model": self.model,
            "base_url": self.base_url
        }


class PoliticalMemeEvaluator:
    """
    Ollama evaluator for political meme analysis.
    Uses <<END_THINK>> marker to separate reasoning from final decision.
    MCQ questions are evaluated by direct comparison; open-ended questions use LLM-as-judge.
    """

    THINK_MARKER = "<<END_THINK>>"

    # Few-shot examples for political meme evaluation
    EXAMPLE_EVALUATION = """
EXAMPLE:
Model Answer: This image most likely relates to the political context of Bangladesh, given the mention of "BANGLADESH" at the top.
Ground Truth: Bangladesh
Analysis: The model answer explicitly identifies Bangladesh as the political context, which matches the ground truth. The additional explanation about how it reached this conclusion does not change the core answer.
<<END_THINK>>
{"decision":"YES", "confidence":0.95, "short_reason":"Both identify Bangladesh as the political context"}
DECISION: YES

EXAMPLE:
Model Answer: The image does not show a clear political bias favoring a particular party, ideology, or individual.
Ground Truth: Yes, the image shows strong political bias against former Prime Minister Sheikh Hasina and her party, the Awami League.
Analysis: The model answer claims no political bias, while the ground truth identifies strong bias against Sheikh Hasina and the Awami League. These are fundamentally contradictory assessments.
<<END_THINK>>
{"decision":"NO", "confidence":0.95, "short_reason":"Model says no bias, but ground truth identifies clear anti-Hasina/Awami League bias"}
DECISION: NO

EXAMPLE:
Model Answer: Yes, the image references political events in Bangladesh. 'Nirbachon' means 'election' in Bengali. This likely refers to the political discourse surrounding the 12th Bangladeshi general election held on January 7, 2024.
Ground Truth: Yes, this image references the contemporary political landscape of Bangladesh following the events of August 5, 2024. It mocks the repetitiveness of a party's rhetoric demanding immediate elections.
Analysis: Both confirm the image references political events. However, the model links it to the January 2024 election, while the ground truth links it to post-August 2024 events. The timeline and context differ significantly.
<<END_THINK>>
{"decision":"NO", "confidence":0.85, "short_reason":"Wrong timeline - model says Jan 2024 election, ground truth says post-Aug 2024 events"}
DECISION: NO

EXAMPLE:
Model Answer: The punchline lies in the reinterpretation of 'NNN' from 'No Nut November' to 'Nirbachon, Nirbachon, Nirbachon' (Election, Election, Election). It humorously captures the exhausting political discourse around elections.
Ground Truth: The punchline lies in the satirical reinterpretation of 'NNN'. The meme redefines it as Nirbachon, Nirbachon, Nirbachon. Since August 2024, Mirza Fakhrul has consistently demanded elections. This meme captures that repetitiveness.
Analysis: Both answers correctly identify the NNN → Nirbachon wordplay as the core punchline. The model captures the humor of repetitive election talk. However, the model misses the specific target (Mirza Fakhrul / BNP) and the post-August 2024 political context. The core punchline mechanism is correctly identified but key political specifics are missing.
<<END_THINK>>
{"decision":"NO", "confidence":0.70, "short_reason":"Core punchline mechanism correct but misses specific political target (Mirza Fakhrul/BNP) and correct timeline"}
DECISION: NO
"""

    def __init__(self, api_manager: OllamaApiManager, seed=42):
        self.api_manager = api_manager
        self.seed = seed
        self.model_name = "gpt-oss:20b"

        logging.info(f"Political meme evaluator initialized with model: {self.model_name}")
        print(f"Political meme evaluator initialized with model: {self.model_name}")

    def generate_response(self, prompt):
        """
        Generate response using Ollama API and split at THINK_MARKER.
        Returns: (thinking_text, final_text, full_response_text)
        """
        try:
            response = self.api_manager.generate_content(
                prompt=prompt,
                timeout=300
            )

            full_response = (response or "").strip()

            # Try to find the last occurrence of the THINK_MARKER
            marker = self.THINK_MARKER
            idx = full_response.rfind(marker)
            if idx >= 0:
                thinking = full_response[:idx].strip()
                final = full_response[idx + len(marker):].strip()
                return thinking, final, full_response

            return "", full_response, full_response

        except Exception as e:
            logging.error(f"Error generating response with Ollama: {e}", exc_info=True)
            return "", f"Error: {str(e)}", ""


def _extract_json_block(text):
    """
    Find the last valid JSON object in text using improved pattern matching.
    Returns (json_obj, json_text) or (None, None)
    """
    if not text:
        return None, None

    # First try: regex pattern for JSON-like structures
    json_pattern = r'\{[^{}]*(?:\{[^{}]*\}[^{}]*)*\}'
    matches = re.findall(json_pattern, text)

    for match in reversed(matches):
        try:
            obj = json.loads(match)
            return obj, match
        except:
            continue

    # Fallback: greedy approach
    last_open = text.rfind('{')
    if last_open == -1:
        return None, None

    candidate = text[last_open:]
    brace_count = 0

    for i, char in enumerate(candidate):
        if char == '{':
            brace_count += 1
        elif char == '}':
            brace_count -= 1
            if brace_count == 0:
                try:
                    maybe = candidate[:i+1]
                    obj = json.loads(maybe)
                    return obj, maybe
                except:
                    continue

    return None, None


def extract_decision_and_confidence(final_text):
    """
    Extract a decision (YES/NO) and optional confidence from the final_text.
    Returns: (decision_str_or_none, confidence_or_none, short_reason_or_none)
    """
    if not final_text:
        return None, None, None

    # 1) Try JSON block first
    obj, obj_text = _extract_json_block(final_text)
    if obj is not None:
        decision = None
        conf = None
        short = None

        for key in ("decision", "Decision", "DECISION"):
            if key in obj:
                decision_val = str(obj[key]).strip().upper()
                if decision_val in ("YES", "NO", "TRUE", "FALSE", "1", "0"):
                    decision = "YES" if decision_val in ("YES", "TRUE", "1") else "NO"
                break

        for key in ("confidence", "conf", "confidence_score"):
            if key in obj:
                try:
                    conf_val = float(obj[key])
                    if 0.0 <= conf_val <= 1.0:
                        conf = conf_val
                    elif 0.0 <= conf_val <= 100.0:
                        conf = conf_val / 100.0
                except:
                    conf = None
                break

        for key in ("short_reason", "reason", "short_reasoning"):
            if key in obj:
                short = str(obj[key])
                break

        if decision in ("YES", "NO"):
            return decision.lower(), conf, short

    # 2) Check ALL lines for DECISION: YES/NO pattern (not just the last line)
    lines = [ln.strip() for ln in final_text.strip().splitlines() if ln.strip()]

    decision_re = re.compile(r'\bDECISION\s*[:=]\s*(YES|NO)\b', re.IGNORECASE)
    for line in reversed(lines):
        m = decision_re.search(line)
        if m:
            return m.group(1).lower(), None, None

    # Check for bare YES/NO lines
    bare_re = re.compile(r'^(YES|NO)[.!?]*$', re.IGNORECASE)
    for line in reversed(lines):
        m = bare_re.match(line)
        if m:
            return m.group(1).lower(), None, None

    # 3) Heuristic matching for political context
    tail = " ".join(lines[-3:]).upper() if lines else ""
    conclusion_indicators = ["THEREFORE", "THUS", "CONCLUSION", "FINAL", "RESULT", "OVERALL"]

    has_conclusion_context = any(indicator in tail for indicator in conclusion_indicators)

    if has_conclusion_context:
        positive_phrases = [
            "SEMANTICALLY EQUIVALENT", "SAME MEANING", "CAPTURES THE SAME",
            "CORRECTLY IDENTIFIES", "MATCHES THE GROUND TRUTH", "ALIGNS WITH"
        ]
        negative_phrases = [
            "NOT EQUIVALENT", "DOES NOT MATCH", "FAILS TO CAPTURE",
            "CONTRADICTS", "MISSES THE KEY", "INCORRECT", "FUNDAMENTALLY DIFFERENT"
        ]

        if any(phrase in tail for phrase in positive_phrases):
            return 'yes', None, None
        if any(phrase in tail for phrase in negative_phrases):
            return 'no', None, None

    # 4) Scan full text for strong positive/negative signals
    full_upper = final_text.upper()
    strong_positive = [
        "BOTH IDENTIFY", "BOTH MENTION", "BOTH SAY", "ANSWER IS CORRECT",
        "MODEL IS CORRECT", "ANSWERS MATCH", "CORRECT COUNTRY",
        "MODEL ANSWER CORRECTLY", "CORRECTLY IDENTIFIES", "SAME COUNTRY",
        "SEMANTICALLY EQUIVALENT", "CORE MEANING IS THE SAME",
        "MATCHES THE GROUND TRUTH", "ALIGNS WITH THE GROUND TRUTH",
        "CORRECTLY STATES", "MODEL CAPTURES", "CAPTURES THE ESSENTIAL",
    ]
    strong_negative = [
        "WRONG COUNTRY", "INCORRECT COUNTRY", "MODEL IS WRONG",
        "DOES NOT MATCH", "ANSWER IS INCORRECT", "DIFFERENT COUNTRY",
        "MISSING KEY", "PARTIALLY CORRECT", "FAILS TO IDENTIFY",
        "FUNDAMENTALLY DIFFERENT", "CONTRADICTS", "INCORRECT ANSWER",
        "MODEL INCORRECTLY", "WRONG ANSWER", "MISIDENTIFIES",
    ]
    for phrase in strong_positive:
        if phrase in full_upper:
            return 'yes', None, None
    for phrase in strong_negative:
        if phrase in full_upper:
            return 'no', None, None

    # 5) For very short responses (1-5 lines), infer from overall content
    if len(lines) <= 5 and lines:
        entity_mismatch = [
            "DIFFERENT", "WRONG", "INCORRECT", "NOT THE SAME", "DOES NOT MATCH",
            "MISMATCH", "CONTRADICT",
        ]
        entity_match = [
            "SAME", "MATCH", "CORRECT", "AGREE", "EQUIVALENT", "ALIGNS",
        ]
        has_match = any(word in full_upper for word in entity_match)
        has_mismatch = any(word in full_upper for word in entity_mismatch)
        if has_match and not has_mismatch:
            return 'yes', None, None
        if has_mismatch and not has_match:
            return 'no', None, None

    return None, None, None


def evaluate_mcq(model_answer, correct_answer):
    """
    Evaluate MCQ questions by direct letter comparison.
    Returns: dict with decision, confidence, short_reason
    """
    model_clean = str(model_answer).strip().upper()
    correct_clean = str(correct_answer).strip().upper()

    is_correct = model_clean == correct_clean

    return {
        'thinking': '',
        'final_text': f"MCQ comparison: model='{model_clean}' vs correct='{correct_clean}'",
        'full_decoded': '',
        'decision': 'yes' if is_correct else 'no',
        'confidence': 1.0,
        'short_reason': f"{'Match' if is_correct else 'Mismatch'}: model={model_clean}, correct={correct_clean}",
    }


def evaluate_open_ended(evaluator, model_answer, correct_answer, question_text, image_id, question_id, max_retries=3):
    """
    Evaluate open-ended questions using Ollama as a semantic judge.
    Returns: dict with thinking, final_text, decision, confidence, short_reason
    """
    model_answer_text = str(model_answer) if model_answer else "No answer provided"
    correct_answer_text = str(correct_answer) if correct_answer else "No answer provided"

    if not model_answer_text.strip() or not correct_answer_text.strip():
        return {
            'thinking': '',
            'final_text': 'Missing essential data',
            'full_decoded': '',
            'decision': 'no',
            'confidence': None,
            'short_reason': 'Missing essential data',
        }

    prompt = f"""You are an expert evaluator for political meme analysis, specializing in Bangladeshi political context. Your task is to determine if the model's answer is semantically equivalent to the ground truth answer.

EVALUATION CRITERIA:

1) FACTUAL ACCURACY: The model answer must identify the same key facts, political entities (parties, leaders, institutions), events, and timelines as the ground truth. Getting the wrong political figure, party, or event timeline is a significant error.

2) POLITICAL CONTEXT: The model must demonstrate understanding of the correct political context. For Bangladeshi political memes, this includes knowledge of:
   - Key political figures (e.g., Sheikh Hasina, Mirza Fakhrul, BNP, Awami League, etc.)
   - Major political events (e.g., 2024 mass uprising, elections, political crises)
   - Political dynamics (party rivalries, alliances, public sentiment)

3) SENTIMENT & BIAS ALIGNMENT: If the ground truth identifies a specific political bias, sentiment, or target, the model must capture the same direction. Saying "no bias" when there is clear bias, or identifying the wrong target, is incorrect.

4) CORE MEANING: For humor/punchline explanations, the model must capture the same core comedic mechanism and political commentary. Identifying the right joke structure but wrong political target or context is NOT sufficient.

5) COMPLETENESS: The model answer does not need to be word-for-word identical, but it must capture the essential claims of the ground truth. Additional correct details are acceptable; missing key information is not.

6) TOLERANCE: Minor differences in wording, phrasing, or additional context that don't change the core meaning are acceptable. The model may provide more detail than the ground truth as long as the core facts align.

7) STRICTNESS ON KEY ELEMENTS: Be strict about:
   - Correct identification of political figures and parties
   - Correct event timelines
   - Correct identification of political bias direction
   - Correct understanding of satirical targets

{PoliticalMemeEvaluator.EXAMPLE_EVALUATION}

EVALUATION PROCESS:
Step 1: Identify what the question is asking (country identification, event reference, target, bias, humor, etc.)
Step 2: Extract the key claims from both the model answer and ground truth.
Step 3: Compare each key claim for factual alignment.
Step 4: Assess whether the model captures the same political understanding.
Step 5: Make final decision — YES if the core meaning and key facts align, NO if there are significant factual errors, wrong targets, or contradictory assessments.

OUTPUT REQUIREMENTS (follow exactly):
1) Give a concise step-by-step analysis showing your reasoning.
2) THEN insert the THINKING marker on its own line exactly:
{PoliticalMemeEvaluator.THINK_MARKER}
3) After the marker, provide the final machine-friendly JSON block like:
   {{"decision":"YES" or "NO", "confidence":0.0-1.0, "short_reason":"..."}}
4) Finally end the entire response with a single final line (no trailing text):
   DECISION: YES
   OR
   DECISION: NO

Now evaluate the following:

IMAGE: {image_id}
QUESTION: {question_text}

MODEL ANSWER:
{model_answer_text}

GROUND TRUTH:
{correct_answer_text}
"""

    last_result = None

    for attempt in range(1, max_retries + 1):
        thinking, final_text, full = evaluator.generate_response(prompt)
        decision, confidence, short_reason = extract_decision_and_confidence(final_text)

        # Fallback: if still no decision, try searching the full raw response
        if decision not in ('yes', 'no') and full:
            decision, confidence, short_reason = extract_decision_and_confidence(full)
            if decision in ('yes', 'no'):
                logging.info(f"{image_id} {question_id}: Decision found in full response (not final_text)")

        result = {
            'thinking': thinking,
            'final_text': final_text,
            'full_decoded': full,
            'decision': decision,
            'confidence': confidence,
            'short_reason': short_reason,
        }
        last_result = result

        if decision in ('yes', 'no'):
            if confidence is not None:
                logging.info(f"{image_id} {question_id}: Got JSON decision with confidence {confidence}")
            return result

        logging.warning(
            f"Attempt {attempt}/{max_retries} failed to get decision for {image_id} {question_id}. "
            f"Raw final_text (first 300 chars): {repr(final_text[:300])}"
        )
        if attempt < max_retries:
            time.sleep(0.1)

    # Default conservatively to 'no'
    logging.error(f"All {max_retries} attempts failed for {image_id} {question_id}; defaulting to 'no'")
    if last_result is None:
        return {
            'thinking': '',
            'final_text': 'All attempts failed - no output',
            'full_decoded': '',
            'decision': 'no',
            'confidence': None,
            'short_reason': 'Evaluation failed',
        }

    last_result['decision'] = 'no'
    if not last_result.get('short_reason'):
        last_result['short_reason'] = 'Failed to get clear decision'
    return last_result


def save_json_atomic(obj, path):
    """Save JSON file atomically to prevent corruption."""
    tmp = path + '.tmp'
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)
    os.replace(tmp, path)


def calculate_confidence_stats(results_list):
    """Calculate statistics for confidence scores from a flat list of question results."""
    confidences = [r.get('confidence') for r in results_list if r.get('confidence') is not None]
    if not confidences:
        return {"count": 0}

    return {
        "count": len(confidences),
        "mean": round(sum(confidences) / len(confidences), 4),
        "min": round(min(confidences), 4),
        "max": round(max(confidences), 4),
        "median": round(sorted(confidences)[len(confidences) // 2], 4)
    }


def main():
    """Main function to execute the political meme evaluation process using Ollama (gpt-oss:20b)."""

    # Configuration — Kaggle paths
    # Input: results from test.py (gemma3:12b evaluation)
    JSON_FILE_PATH = '/kaggle/working/ollama_qwen35_results_20260414_110929.json'
    OUTPUT_FILE_PATH = '/kaggle/working/political_evaluation_results3.json'

    MAX_RETRIES = 3

    # Initialize Ollama API manager and evaluator
    ollama_base_url = os.environ.get("OLLAMA_BASE_URL", "http://localhost:11434")
    
    try:
        logging.info(f"Initializing Ollama with model: gpt-oss:20b")
        api_manager = OllamaApiManager(
            base_url=ollama_base_url,
            model="gpt-oss:20b",
            rate_limit_delay=2
        )
        logging.info("Ollama initialized successfully")

    except Exception as e:
        logging.error(f"Failed to initialize Ollama: {e}")
        print(f"Error initializing Ollama: {e}")
        print("\nFor Kaggle setup:")
        print("1. Install Ollama: !curl -fsSL https://ollama.com/install.sh | sh")
        print("2. Start Ollama: import subprocess; subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)")
        print("3. Pull model: !ollama pull gpt-oss:20b")
        print("4. Then run this script")
        return

    evaluator = PoliticalMemeEvaluator(
        api_manager=api_manager,
        seed=42
    )

    # Load input data
    try:
        with open(JSON_FILE_PATH, 'r', encoding='utf-8') as f:
            json_data = json.load(f)
        logging.info(f"Loaded {len(json_data)} images from JSON file")
        print(f"Loaded {len(json_data)} images from JSON file")
    except Exception as e:
        logging.error(f"Failed to load JSON input: {e}")
        print(f"Error loading JSON: {e}")
        return

    # Tracking variables
    image_evaluations = []
    all_question_results = []  # flat list for stats

    # Per-question-id tracking
    per_question_correct = {}   # Q1..Q7 -> count correct
    per_question_total = {}     # Q1..Q7 -> count total

    # Per-type tracking
    mcq_correct = 0
    mcq_total = 0
    open_ended_correct = 0
    open_ended_total = 0

    total_questions = 0
    total_correct = 0
    total_images = len(json_data)

    # Process each image
    for img_idx, image_entry in enumerate(tqdm(json_data, desc='Evaluating political memes')):
    # for img_idx, image_entry in enumerate(tqdm(json_data[:2], desc='Evaluating political memes')):
        try:
            image_id = image_entry.get('image_id', f'IMAGE_{img_idx+1:03d}')
            status = image_entry.get('status', 'unknown')
            questions = image_entry.get('questions', [])

            if status != 'success' or not questions:
                logging.warning(f"Skipping {image_id}: status={status}, questions={len(questions)}")
                image_evaluations.append({
                    'image_id': image_id,
                    'status': status,
                    'questions': [],
                    'image_accuracy': 0.0,
                    'note': 'Skipped - no successful results'
                })
                continue

            image_question_results = []
            image_correct = 0

            for q in questions:
                question_id = q.get('question_id', 'unknown')
                question_type = q.get('question_type', 'open_ended')
                question_text = q.get('question_text', '')
                model_answer = q.get('model_answer', '')
                correct_answer = q.get('correct_answer', '')
                options = q.get('options', None)

                # Initialize per-question tracking
                if question_id not in per_question_correct:
                    per_question_correct[question_id] = 0
                    per_question_total[question_id] = 0

                # Evaluate based on question type
                if question_type == 'mcq':
                    result = evaluate_mcq(model_answer, correct_answer)
                    mcq_total += 1
                    if result['decision'] == 'yes':
                        mcq_correct += 1
                else:
                    result = evaluate_open_ended(
                        evaluator, model_answer, correct_answer,
                        question_text, image_id, question_id,
                        max_retries=MAX_RETRIES
                    )
                    open_ended_total += 1
                    if result['decision'] == 'yes':
                        open_ended_correct += 1

                # Build question result entry
                q_result = {
                    'question_id': question_id,
                    'question_type': question_type,
                    'question_text': question_text,
                    'model_answer': model_answer,
                    'correct_answer': correct_answer,
                    'decision': result['decision'],
                    'confidence': result.get('confidence'),
                    'short_reason': result.get('short_reason'),
                    'thinking': result.get('thinking', ''),
                    'final_text': result.get('final_text', ''),
                }
                if options:
                    q_result['options'] = options

                image_question_results.append(q_result)
                all_question_results.append(q_result)

                # Update counters
                total_questions += 1
                per_question_total[question_id] += 1

                if result['decision'] == 'yes':
                    total_correct += 1
                    image_correct += 1
                    per_question_correct[question_id] += 1

                # Print progress
                decision_str = result['decision'].upper() if result['decision'] else 'UNKNOWN'
                conf_str = f" (conf: {result['confidence']:.2f})" if result.get('confidence') else ""
                print(f"  {image_id} {question_id} [{question_type}]: {decision_str}{conf_str}")

            # Image-level accuracy
            image_acc = (image_correct / len(questions)) * 100 if questions else 0.0

            image_evaluations.append({
                'image_id': image_id,
                'status': status,
                'questions': image_question_results,
                'image_correct': image_correct,
                'image_total': len(questions),
                'image_accuracy': round(image_acc, 2),
            })

            logging.info(f"{image_id}: {image_correct}/{len(questions)} correct ({image_acc:.1f}%)")

            # Intermediate save every 10 images
            if (img_idx + 1) % 10 == 0:
                overall_acc = (total_correct / total_questions) * 100 if total_questions else 0
                logging.info(f"Progress: {img_idx+1}/{total_images} images, "
                           f"Overall: {total_correct}/{total_questions} ({overall_acc:.2f}%)")
                intermediate_file = OUTPUT_FILE_PATH.replace('.json', f'_checkpoint_{img_idx+1}.json')
                save_json_atomic(image_evaluations, intermediate_file)

        except Exception as e:
            logging.error(f"Error processing {image_entry.get('image_id', img_idx)}: {e}", exc_info=True)
            image_evaluations.append({
                'image_id': image_entry.get('image_id', f'IMAGE_{img_idx+1:03d}'),
                'status': 'error',
                'questions': [],
                'image_accuracy': 0.0,
                'note': f'Error during evaluation: {str(e)}'
            })
            continue

    # Calculate final statistics
    overall_accuracy = (total_correct / total_questions) * 100 if total_questions else 0
    mcq_accuracy = (mcq_correct / mcq_total) * 100 if mcq_total else 0
    open_ended_accuracy = (open_ended_correct / open_ended_total) * 100 if open_ended_total else 0
    confidence_stats = calculate_confidence_stats(all_question_results)

    # Per-question accuracy
    per_question_accuracy = {}
    for qid in sorted(per_question_correct.keys()):
        total = per_question_total[qid]
        correct = per_question_correct[qid]
        per_question_accuracy[qid] = {
            'correct': correct,
            'total': total,
            'accuracy': round((correct / total) * 100, 2) if total else 0.0
        }

    # Prepare final results
    final_results = {
        'evaluation_metadata': {
            'model_used': evaluator.model_name,
            'total_images': total_images,
            'total_questions': total_questions,
            'total_correct': total_correct,
            'overall_accuracy_percentage': round(overall_accuracy, 2),
            'mcq_stats': {
                'total': mcq_total,
                'correct': mcq_correct,
                'accuracy_percentage': round(mcq_accuracy, 2)
            },
            'open_ended_stats': {
                'total': open_ended_total,
                'correct': open_ended_correct,
                'accuracy_percentage': round(open_ended_accuracy, 2)
            },
            'per_question_accuracy': per_question_accuracy,
            'confidence_statistics': confidence_stats,
            'evaluation_timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
            'settings': {
                'max_retries': MAX_RETRIES,
                'seed': evaluator.seed,
            }
        },
        'image_evaluations': image_evaluations,
    }

    # Save final results
    save_json_atomic(final_results, OUTPUT_FILE_PATH)

    # Print summary
    print('\n' + '='*80)
    print('POLITICAL MEME EVALUATION COMPLETED!')
    print('='*80)
    print(f"Judge Model Used: {evaluator.model_name} (Ollama)")
    print(f"Total Images: {total_images}")
    print(f"Total Questions: {total_questions}")
    print(f"Total Correct: {total_correct}")
    print(f"Overall Accuracy: {overall_accuracy:.2f}%")
    print(f"\nMCQ Accuracy: {mcq_correct}/{mcq_total} ({mcq_accuracy:.2f}%)")
    print(f"Open-Ended Accuracy: {open_ended_correct}/{open_ended_total} ({open_ended_accuracy:.2f}%)")
    print(f"\nPer-Question Accuracy:")
    for qid, stats in per_question_accuracy.items():
        print(f"  {qid}: {stats['correct']}/{stats['total']} ({stats['accuracy']:.2f}%)")
    if confidence_stats.get('count', 0) > 0:
        print(f"\nConfidence Stats (open-ended):")
        print(f"  Average: {confidence_stats['mean']:.3f}")
        print(f"  Range: {confidence_stats['min']:.3f} - {confidence_stats['max']:.3f}")
    print(f"\nResults saved to: {OUTPUT_FILE_PATH}")
    print('='*80)

    # Print Ollama usage stats
    usage_stats = api_manager.get_usage_stats()
    print(f"\nOllama API Calls Made: {usage_stats['total_calls']}")

    # Final logging
    logging.info('Political meme evaluation completed!')
    logging.info(f"Final Results: {total_correct}/{total_questions} correct ({overall_accuracy:.2f}%)")
    logging.info(f"MCQ: {mcq_correct}/{mcq_total} ({mcq_accuracy:.2f}%)")
    logging.info(f"Open-ended: {open_ended_correct}/{open_ended_total} ({open_ended_accuracy:.2f}%)")
    logging.info(f"Results saved to: {OUTPUT_FILE_PATH}")


if __name__ == '__main__':
    main()

2026-04-14 11:21:19,747 - INFO - Initializing Ollama with model: gpt-oss:20b
2026-04-14 11:21:19,748 - INFO - Ollama API Manager initialized: http://localhost:11434, model=gpt-oss:20b
2026-04-14 11:21:19,753 - INFO - Available models: ['gemma3:12b', 'gpt-oss:20b', 'qwen3-vl:8b']
2026-04-14 11:21:19,753 - INFO - Ollama initialized successfully
2026-04-14 11:21:19,754 - INFO - Political meme evaluator initialized with model: gpt-oss:20b
2026-04-14 11:21:19,756 - INFO - Loaded 20 images from JSON file


Political meme evaluator initialized with model: gpt-oss:20b
Loaded 20 images from JSON file


Evaluating political memes:   0%|          | 0/20 [00:00<?, ?it/s]

  IMAGE_001 Q1 [open_ended]: YES


2026-04-14 11:22:18,795 - INFO - IMAGE_001 Q2: Got JSON decision with confidence 0.95


  IMAGE_001 Q2 [open_ended]: NO (conf: 0.95)
  IMAGE_001 Q3 [mcq]: YES (conf: 1.00)
  IMAGE_001 Q4 [mcq]: YES (conf: 1.00)
  IMAGE_001 Q5 [mcq]: YES (conf: 1.00)


... (124 outputs left out)

2026-04-14 11:34:55,328 - INFO - IMAGE_020 Q6: Got JSON decision with confidence 0.98


  IMAGE_020 Q6 [open_ended]: NO (conf: 0.98)


2026-04-14 11:35:04,997 - INFO - IMAGE_020 Q7: Got JSON decision with confidence 0.98
2026-04-14 11:35:04,999 - INFO - IMAGE_020: 2/7 correct (28.6%)
2026-04-14 11:35:05,000 - INFO - Progress: 20/20 images, Overall: 41/140 (29.29%)
Evaluating political memes: 100%|██████████| 20/20 [13:45<00:00, 41.26s/it]
2026-04-14 11:35:05,013 - INFO - Political meme evaluation completed!
2026-04-14 11:35:05,013 - INFO - Final Results: 41/140 correct (29.29%)
2026-04-14 11:35:05,014 - INFO - MCQ: 20/60 (33.33%)
2026-04-14 11:35:05,015 - INFO - Open-ended: 21/80 (26.25%)
2026-04-14 11:35:05,016 - INFO - Results saved to: /kaggle/working/political_evaluation_results3.json


  IMAGE_020 Q7 [open_ended]: NO (conf: 0.98)

POLITICAL MEME EVALUATION COMPLETED!
Judge Model Used: gpt-oss:20b (Ollama)
Total Images: 20
Total Questions: 140
Total Correct: 41
Overall Accuracy: 29.29%

MCQ Accuracy: 20/60 (33.33%)
Open-Ended Accuracy: 21/80 (26.25%)

Per-Question Accuracy:
  Q1: 20/20 (100.00%)
  Q2: 0/20 (0.00%)
  Q3: 7/20 (35.00%)
  Q4: 2/20 (10.00%)
  Q5: 11/20 (55.00%)
  Q6: 1/20 (5.00%)
  Q7: 0/20 (0.00%)

Confidence Stats (open-ended):
  Average: 0.984
  Range: 0.900 - 1.000

Results saved to: /kaggle/working/political_evaluation_results3.json

Ollama API Calls Made: 80
